# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
!pip uninstall -y vllm torch torchvision torchaudio xformers
!pip install -U uv
!uv pip install --system --no-cache vllm==0.19.1
!uv pip install --system --no-cache transformers==4.57.6 tqdm sympy numpy bitsandbytes antlr4-python3-runtime==4.11.1 accelerate

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 101.7 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 1.44s
Prepared 55 packages in 23.86s
Uninstalled 5 packages in 386ms
Installed 55 packages in 264ms
 + anthropic==0.102.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==6.1.1
 + compressed-tensors==0.15.0.1
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.1
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 +

In [2]:
!uv pip install --system --no-cache peft trl datasets accelerate

Using Python 3.12.13 environment at: /usr
Resolved 69 packages in 795ms
Prepared 3 packages in 768ms
Uninstalled 2 packages in 38ms
Installed 3 packages in 16ms
 - datasets==4.0.0
 + datasets==4.8.5
 - pyarrow==18.1.0
 + pyarrow==24.0.0
 + trl==1.4.0


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [8]:
import os
import json
import random
import re
import sys
from pathlib import Path

import torch
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

DATA_PATH = "/content/151B_SP26_Competition/data/public.jsonl"      # change if needed
OUTPUT_DIR = "qwen3_4b_math_lora"
FINAL_LORA_DIR = "qwen3_4b_math_lora_final"

SEED = 151
N_SAMPLE = 100
MAX_TOKENS = 32768

random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__)

torch: 2.10.0+cu128


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [6]:
!git clone https://github.com/Atakamoto/151B_SP26_Competition.git

Cloning into '151B_SP26_Competition'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 61 (delta 24), reused 8 (delta 8), pack-reused 27 (from 2)
Receiving objects: 100% (61/61), 376.52 KiB | 4.77 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [9]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)

print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form)")

rng = random.Random(SEED)
sample_indices = rng.sample(range(len(data)), k=min(N_SAMPLE, len(data)))

sample_data = [data[i] for i in sample_indices]
sample_ids = [item["id"] for item in sample_data]
eval_ids = set(sample_ids)

train_public_data = [item for item in data if item["id"] not in eval_ids]

print("Eval sample:", len(sample_data))
print("Public training examples:", len(train_public_data))
print("First 10 eval IDs:", sample_ids[:10])

Loaded 1126 questions (375 MCQ, 751 free-form)
Eval sample: 100
Public training examples: 1026
First 10 eval IDs: [838, 419, 721, 578, 756, 237, 944, 1002, 151, 937]


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [11]:
SYSTEM_PROMPT_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Please reason step by step and solve the problem carefully. "
    "Put your final answer within \\boxed{}. "
    "If the problem has multiple [ANS] placeholders, give the answers in the same order, "
    "separated by commas inside a single \\boxed{}, e.g. \\boxed{3, 7}. "
    "Do not write [ANS] in your final answer."
    "Do not round decimal answers unless the problem explicitly asks for rounding."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician solving a multiple-choice math problem. "
    "Think carefully to determine the correct option. "
    "Do not put intermediate results in \\boxed{}. "
    "At the very end, output only one boxed capital letter. "
    "Do not include any explanation after the final answer. "
    "Example final format: \\boxed{C}."
)

SYSTEM_PROMPT_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
#     return SYSTEM_PROMPT_MATH, question
def build_prompt(question, options=None):
    if options is None:
        num_ans = question.count("[ANS]")
        system = SYSTEM_PROMPT_MATH
        user = (
            f"{question}\n\n"
            f"There are {num_ans} [ANS] placeholder(s). "
            f"Give exactly {num_ans} final answer(s), in order, inside one final \\boxed{{}}."
        )
    else:
        system = SYSTEM_PROMPT_MCQ
        choices = "\n".join(
            f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)
        )
        user = (
            f"{question}\n\n"
            f"Answer choices:\n{choices}\n\n"
            "Choose exactly one option letter."
        )
    return system, user

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [13]:
def gold_to_boxed(answer):
    if isinstance(answer, list):
        ans = ", ".join(str(x) for x in answer)
    else:
        ans = str(answer)
    return f"\\boxed{{{ans}}}"


def make_competition_training_text(item):
    system, user = build_prompt(item["question"], item.get("options"))

    # These public examples mostly teach formatting, not deep reasoning.
    assistant = (
        "We solve the problem carefully and give the final answer in the required format.\n\n"
        f"{gold_to_boxed(item['answer'])}"
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def get_first_available(ex, keys):
    for k in keys:
        if k in ex and ex[k]:
            return ex[k]
    return None


def make_openmath_text(ex):
    question = get_first_available(
        ex,
        ["problem", "question", "input", "prompt"]
    )
    solution = get_first_available(
        ex,
        ["generated_solution", "solution", "output", "response", "completion"]
    )

    if question is None or solution is None:
        return None

    question = str(question)
    solution = str(solution)

    # Keep examples that teach the final answer format.
    if "\\boxed" not in solution:
        return None

    # Avoid training on super long rambling examples.
    # OpenMathInstruct-2 specifically found overly verbose solutions can hurt.
    if len(solution) > 6000:
        return None

    system = (
        "You are an expert mathematician. "
        "Please reason step by step and solve the problem carefully. "
        "Put your final answer within \\boxed{}."
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
        {"role": "assistant", "content": solution},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [14]:
OPENMATH_TARGET = 4000
PUBLIC_TARGET = 1000

openmath_stream = load_dataset(
    "nvidia/OpenMathInstruct-2",
    split="train",
    streaming=True,
)

openmath_texts = []

for ex in openmath_stream:
    text = make_openmath_text(ex)
    if text is not None:
        openmath_texts.append(text)

    if len(openmath_texts) >= OPENMATH_TARGET:
        break

print("OpenMath examples:", len(openmath_texts))
print(openmath_texts[0][:1000])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

OpenMath examples: 4000
<|im_start|>system
You are an expert mathematician. Please reason step by step and solve the problem carefully. Put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Ava is planning a camping trip with her friends. She wants to make sure they have enough granola bars for snacks. There will be five people total: Ava, her two friends, and her parents. They will spend 3 days and 2 nights at the campsite, and they plan to have 2 granola bars per person for breakfast and 1 granola bar per person for an afternoon snack each day. How many granola bars will Ava need to pack in total for the entire trip?<|im_end|>
<|im_start|>assistant
<think>

</think>

There will be a total of 5 people.
Each person needs 2 granola bars for breakfast and 1 granola bar for snack. This amounts to a total of 3 granola bars per person per day.
Since the trip is 3 days long, each person will need 3 granola bars/day * 3 days = 9 granola bars.
So for 5 people, Ava will need 5 * 9 =

In [15]:
competition_texts = [
    make_competition_training_text(item)
    for item in train_public_data
]

random.shuffle(competition_texts)

train_texts = []
train_texts.extend(openmath_texts[:OPENMATH_TARGET])
train_texts.extend(competition_texts[:min(PUBLIC_TARGET, len(competition_texts))])

random.shuffle(train_texts)

train_dataset = Dataset.from_dict({"text": train_texts})

print(train_dataset)
print("Total training examples:", len(train_dataset))
print(train_dataset[0]["text"][:1000])

Dataset({
    features: ['text'],
    num_rows: 5000
})
Total training examples: 5000
<|im_start|>system
You are an expert mathematician. Please reason step by step and solve the problem carefully. Put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Let $S$ be a subset of $\{1, 2, 3, \cdots, 2007\}$, such that for every pair of numbers $x$ and $y$ in $S$, the product $xy$ is not divisible by $29$.  What is the maximum size of $S$?<|im_end|>
<|im_start|>assistant
<think>

</think>

To solve this problem, we need to find the maximum number of elements in $S$ such that for every pair of numbers $x$ and $y$ in $S$, the product $xy$ is not divisible by 29.

Note that $29$ is a prime number, so if $xy$ is divisible by 29, then either $x$ or $y$ must be divisible by 29. There are $2007 : 29 = 69$ numbers in the set $\{1, 2, 3, \cdots, 2007\}$ that are divisible by 29.

To maximize the size of $S$, we need to include as many numbers as possible that are not divisible by 29. There

In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [17]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [19]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=4096,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=250,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    packing=False,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

trainer.save_model(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)

print(f"Saved LoRA adapter to {FINAL_LORA_DIR}")

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,1.189700
20,1.158900
30,1.085800
40,1.015300
50,0.956600
60,0.892000
70,0.785800
80,0.707200
90,0.700100
100,0.625600


Saved LoRA adapter to qwen3_4b_math_lora_final


In [25]:
del trainer
del model
torch.cuda.empty_cache()

In [26]:
llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_lora=True,
    trust_remote_code=True,
    max_model_len=16384,
    gpu_memory_utilization=0.75,
    max_num_seqs=64,
    max_num_batched_tokens=32768,
)

In [20]:
import random

SEED = 151
N_SAMPLE = 100

rng = random.Random(SEED)

# sample fixed indices
sample_indices = rng.sample(range(len(data)), k=min(N_SAMPLE, len(data)))

# fixed sampled data
sample_data = [data[i] for i in sample_indices]
sample_ids = [item["id"] for item in sample_data]

print(f"Sampled {len(sample_data)} questions with seed={SEED}")
print("First 10 sampled ids:", [item["id"] for item in sample_data[:10]])

Sampled 100 questions with seed=151
First 10 sampled ids: [838, 419, 721, 578, 756, 237, 944, 1002, 151, 937]


In [21]:
prompts = []

for item in sample_data:
    system, user = build_prompt(item["question"], item.get("options"))

    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompts.append(prompt_text)

print("Built prompts:", len(prompts))

Built prompts: 100


In [27]:
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest(
        "math_lora",
        1,
        FINAL_LORA_DIR,
    ),
)

responses = [out.outputs[0].text.strip() for out in outputs]

print("Generated:", len(responses))
print(responses[0][:1000])

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generated: 100



In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
!cp -r qwen3_4b_math_lora_final /content/drive/MyDrive/qwen3_4b_math_lora_final

### Generate with vLLM

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in sample_data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 100 questions...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's try to figure out this problem step by step. So, first, Sylvia has a compact car with an 8-gallon tank. The car gets 33 miles per gallon. We need to find out how many miles it can travel when the tank is only 3/4 full. 

First, let me recall that if the tank is full, it can hold 8 gallons, right? So when it's 3/4 full, the amount of fuel in the tank would be 3/4 of 8 gallons. Let me ca ...

── Response 1 (id=1) ──
Okay, let's see. So the problem is about Cody buying 19 roses, and a dozen (which is 12 roses) costs $11.99. We need to find a fair price for 19 roses. Hmm, fair price probably means the exact cost without any discounts or anything, just the total cost based on the price per dozen.

First, let me recall that a dozen is 12 roses. So the cost for 12 roses is $11.99. Now, Cody wants 19 roses. So we n ...

── Response 2 (id=2) ──
Okay, let's try to figure out this problem step by step. The question is to complete the equation: cos(163 - 327) =

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [28]:
%cd 151B_SP26_Competition/

/content/151B_SP26_Competition


In [29]:
!pwd

/content/151B_SP26_Competition


In [30]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(sample_data, responses), total=len(sample_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 100/100 [00:07<00:00, 14.12it/s]

Scoring complete. 100 results.


## 8. Summary

Print accuracy broken down by question type.

In [31]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   31 /   42  (73.81%)
  Free-form  :   18 /   58  (31.03%)
  Overall    :   49 /  100  (49.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [38]:
OUTPUT_PATH = "/content/151B_SP26_Competition/data/first_test.json"

In [39]:
SAVE_EVAL = True  # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to /content/151B_SP26_Competition/data/first_test.json


In [37]:
responses

['',
 '',
 'Okay, let\'s try to figure out this problem step by step. The question is: Complete the following: cos(163 - 327) = [ANS] ( [ANS] ) · cos( [ANS] ) [ANS] ( [ANS] ) · sin( [ANS] ). The angles are in radians. \n\nFirst, let\'s compute 163 - 327. Let me check that: 163 minus 327. Since 327 is bigger than 163, this will be negative. Let\'s calculate it: 327 - 163 = 164, so 163 - 327 = -164 radians. So the left side is cos(-164). \n\nNow, cosine is an even function, so cos(-x) = cos(x). Therefore, cos(-164) = cos(164). So the left side simplifies to cos(164). \n\nBut wait, maybe we need to express this using trigonometric identities, like product-to-sum or sum-to-product? The right side has a product of cosine and sine terms, so perhaps it\'s a form of the sine addition formula or cosine subtraction formula. Let me recall some identities.\n\nThe problem says to complete the equation with the placeholders. Let\'s see. The structure is: cos(A) = [ANS] ( [ANS] ) · cos( [ANS] ) [ANS]

In [ ]:
import csv
from pathlib import Path

OUTPUT_CSV = "results/submission.csv"

rows = []
for item, response in zip(data, responses):
    rows.append({
        "id": item["id"],
        "response": response,
    })

out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

Saved 943 rows to results/submission.csv


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!